# Exercise 8: Customer Who Visited but Did Not Make Any Transactions

**Source:** [Leet Code 1581](https://leetcode.com/problems/customer-who-visited-but-did-not-make-any-transactions/description/?envType=study-plan-v2&envId=top-sql-50) (Easy).

Table: `Visits`
```
+-------------+---------+
| Column Name | Type    |
+-------------+---------+
| visit_id    | int     |
| customer_id | int     |
+-------------+---------+
```
- `visit_id` is the column with unique values.
- `customer_id` is the identity of the customer who visited the mall.
 

Table: `Transactions`
```
+----------------+---------+
| Column Name    | Type    |
+----------------+---------+
| transaction_id | int     |
| visit_id       | int     |
| amount         | int     |
+----------------+---------+
```
- `transaction_id` is column with unique values.
- `visit_id` registers the visit with the respective transaction and the next column its `amount`.

**Task.**
Return the `visit_id`s of the visitors who did not make a transaction and how often they visited. 

Connect to the database `postgres` using the username `postgres`.

In [9]:
%load_ext sql
%sql postgresql+psycopg://postgres@localhost:5432/postgres

The sql extension is already loaded. To reload it, use:
  %reload_ext sql


### Create new database for the exercise

Make sure first that a database with the same name is removed in case it already exists.

In [10]:
%sql DROP DATABASE IF EXISTS exercise_8 WITH (FORCE);
%sql CREATE DATABASE exercise_8;

Running query in 'postgresql+psycopg://postgres@localhost:5432/postgres'

Running query in 'postgresql+psycopg://postgres@localhost:5432/postgres'

++
||
++
++

Connect to the database.

*Caution.* No semicolon at the end otherwise it belongs to the name of the database.

In [11]:
%sql postgresql+psycopg://postgres@localhost:5432/exercise_8 --alias exercise_8

Connecting and switching to connection 'exercise_8'

#### Fill the database with content

In [12]:
%%sql

DROP TABLE IF EXISTS Visits;
DROP TABLE IF EXISTS Transactions;

CREATE TABLE Visits(
    visit_id INTEGER NOT NULL UNIQUE,
    customer_id INTEGER NOT NULL
);

INSERT INTO Visits (visit_id, customer_id)
VALUES
(1, 23),
(2, 9),
(4, 30),
(5, 54),
(6, 96),
(7, 54),
(8, 54);

CREATE TABLE Transactions (
    transaction_id INTEGER NOT NULL UNIQUE,
    visit_id INTEGER NOT NULL,
    amount INTEGER NOT NULL
);

INSERT INTO Transactions (transaction_id, visit_id, amount)
VALUES
(2, 5, 310),
(3, 5, 300),
(9, 5, 200),
(12, 1, 910),
(13, 2, 970);

SELECT * FROM Transactions;

Running query in 'exercise_8'

7 rows affected.

5 rows affected.

5 rows affected.

transaction_id,visit_id,amount
2,5,310
3,5,300
9,5,200
12,1,910
13,2,970


#### Solve the exercise

**Task.**
Return the `visit_id`s of the visitors who did not make a transaction and how often they visited.

In [13]:
%%sql
SELECT Visits.customer_id, COUNT(Visits.visit_id) AS count_no_trans
FROM Visits
WHERE Visits.visit_id NOT IN
(SELECT Transactions.visit_id FROM Transactions)
GROUP BY customer_id;

Running query in 'exercise_8'

3 rows affected.

customer_id,count_no_trans
30,1
54,2
96,1


#### Explanation
- One needs to select `customer_id` from Visits such that the corresponding `visit_id` does not appear in Transactions,
therefore the subquery `SELECT Transactions.visit_id FROM Transactions`
- Then we need to count how often `visit_id` for a given `customer_id`. This can be done via `GROUP BY customer_id` and the function `COUNT(visit_id)`.  

#### Addendum

The solution does not use a `JOIN`.
Here is a solution that solves the exercise by using a `JOIN`.

In [ ]:
%%sql
SELECT V.customer_id, COUNT(V.customer_id) AS count_no_trans
FROM Visits V LEFT JOIN Transactions T ON T.visit_id = V.visit_id
WHERE T.transaction_id IS NULL
GROUP BY 1;

Running query in 'exercise_8'

3 rows affected.

customer_id,count_no_trans
30,1
54,2
96,1


#### Explanation
- By joining the `Visits` via `LEFT JOIN` with `Transactions` on `visit_id`, the visitor who did not make a transaction have `transaction_id NULL`.
- The `GROUP BY` clause works as before. Note that `1` refers to `V.customer_id` (i.e. the first selected column).

#### Close connection

In [15]:
%sql --close exercise_8
%sql postgresql+psycopg://postgres@localhost:5432/postgres
%sql DROP DATABASE IF EXISTS exercise_8;

Switching to connection 'postgresql+psycopg://postgres@localhost:5432/postgres'

Running query in 'postgresql+psycopg://postgres@localhost:5432/postgres'

++
||
++
++

**Well done!** (approved by Leet)